In [2]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
# pandas formatting for data display
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [5]:
#  load data
# convert "week_start" column's data type from strings to "date/time" format
ledger = pd.read_csv("supply_data/inventory_ledger.csv", parse_dates=["week_start"])
stores = pd.read_csv("supply_data/stores.csv")
products = pd.read_csv("supply_data/products.csv")
suppliers = pd.read_csv("supply_data/suppliers.csv")
print(f"Raw inventory_ledger rows: {len(ledger)}")

Raw inventory_ledger rows: 17417


In [8]:
#  Data cleaning
# remove duplicates
# handling missing values
before = len(ledger)
ledger = ledger.drop_duplicates()
print(f"Removed {before - len(ledger)} duplicate rows")
# remove unnecessary space from beginning and end . 
ledger["category"] = ledger["category"].str.strip().str.title()
ledger["units_received"] = ledger["units_received"].fillna(0)

ledger.to_csv("supply_data/inventory_ledger_clean.csv", index=False)
print("Saved cleaned data to supply_data/inventory_ledger_clean.csv")

Removed 0 duplicate rows
Saved cleaned data to supply_data/inventory_ledger_clean.csv


In [9]:
#  merging data 
df = (ledger
      .merge(stores, on="store_id", how="left")
      .merge(products, on="product_id", how="left", suffixes=("", "_prod"))
      .merge(suppliers[["supplier_id", "avg_lead_time_days", "reliability_score"]], on="supplier_id", how="left"))
# if there category is missing in ledger data then fill category from category of product data  
df["category"] = df["category"].fillna(df["category_prod"])

In [11]:
#  core KPIs
total_weeks_rows = len(df)
#  Fill Rate Percentage
overall_fill_rate = 100 * (1 - df["stockout_flag"].mean())
#  Stockout Rate Percentage
overall_stockout_rate = 100 * df["stockout_flag"].mean()
df["lost_sales_value"] = df["lost_sales_units"] * df["unit_price"]
total_lost_sales_value = df["lost_sales_value"].sum()

# inventory turnover (approx): annual COGS / avg inventory value, per SKU
# (per store-product first, then averaged within category -- summing COGS
# across many SKUs and dividing by one row's average inventory value would
# badly inflate the ratio, the same pooling trap as the elasticity calc
# in the pricing project)
df["inventory_value"] = df["closing_stock"] * df["unit_cost"]
# cost of sold products
df["cogs"] = df["units_sold"] * df["unit_cost"]

sku_turnover = df.groupby(["category", "store_id", "product_id"]).apply(
    lambda g: g["cogs"].sum() / g["inventory_value"].mean() if g["inventory_value"].mean() > 0 else np.nan,
    include_groups=False
).rename("turnover").reset_index()

turnover_by_category = sku_turnover.groupby("category")["turnover"].mean().sort_values(ascending=False)
days_of_inventory_by_category = (365 / turnover_by_category).round(1)

print(f"\nOverall fill rate: {overall_fill_rate:.1f}% | Overall stockout rate: {overall_stockout_rate:.1f}%")
print(f"Total estimated lost sales value: {total_lost_sales_value:,.1f}")
print("\n--- Inventory turnover (annualized) by category ---")
print(turnover_by_category.round(2))
print("\n--- Days of inventory outstanding by category ---")
print(days_of_inventory_by_category)



Overall fill rate: 98.2% | Overall stockout rate: 1.8%
Total estimated lost sales value: 2,123,037.8

--- Inventory turnover (annualized) by category ---
category
Household Essentials   24.47
Packaged Foods         19.67
Perishables            12.32
Personal Care          11.37
Beverages              10.90
Name: turnover, dtype: float64

--- Days of inventory outstanding by category ---
category
Household Essentials   14.90
Packaged Foods         18.60
Perishables            29.60
Personal Care          32.10
Beverages              33.50
Name: turnover, dtype: float64


In [12]:
#  Stockouts:
stockout_by_category = df.groupby("category")["stockout_flag"].mean().sort_values(ascending=False) * 100
stockout_by_region = df.groupby("region")["stockout_flag"].mean().sort_values(ascending=False) * 100
stockout_by_store_type = df.groupby("store_type")["stockout_flag"].mean().sort_values(ascending=False) * 100

print("\n--- Stockout rate by category (%) ---")
print(stockout_by_category.round(1))
print("\n--- Stockout rate by region (%) ---")
print(stockout_by_region.round(1))
print("\n--- Stockout rate by store type (%) ---")
print(stockout_by_store_type.round(1))

# supplier performance vs stockouts
supplier_perf = df.groupby("supplier_id").agg(
    avg_lead_time_days=("avg_lead_time_days", "first"),
    reliability_score=("reliability_score", "first"),
    stockout_rate_pct=("stockout_flag", lambda x: 100 * x.mean()),
).round(2).sort_values("stockout_rate_pct", ascending=False)
print("\n--- Supplier performance vs. stockout rate ---")
print(supplier_perf)

#  Correlations 
lead_time_stockout_corr = supplier_perf["avg_lead_time_days"].corr(supplier_perf["stockout_rate_pct"])
reliability_stockout_corr = supplier_perf["reliability_score"].corr(supplier_perf["stockout_rate_pct"])
print(f"\nCorrelation (lead time vs. stockout rate): {lead_time_stockout_corr:.2f}")
print(f"Correlation (reliability vs. stockout rate): {reliability_stockout_corr:.2f}")


--- Stockout rate by category (%) ---
category
Packaged Foods         2.20
Beverages              1.80
Household Essentials   1.80
Perishables            1.60
Personal Care          1.30
Name: stockout_flag, dtype: float64

--- Stockout rate by region (%) ---
region
West    1.90
North   1.90
South   1.60
East    1.60
Name: stockout_flag, dtype: float64

--- Stockout rate by store type (%) ---
store_type
Express    1.90
Flagship   1.90
Standard   1.70
Name: stockout_flag, dtype: float64

--- Supplier performance vs. stockout rate ---
             avg_lead_time_days  reliability_score  stockout_rate_pct
supplier_id                                                          
SUP003                       10               0.85               3.38
SUP007                        6               0.89               2.91
SUP002                       21               0.92               2.33
SUP006                       15               0.83               1.98
SUP005                       19         

In [17]:
# Spoilage by category and Total spoilage
df["spoilage_cost"] = df["spoilage_units"] * df["unit_cost"]
spoilage_by_category = df.groupby("category")["spoilage_cost"].sum().sort_values(ascending=False)
total_spoilage_cost = df["spoilage_cost"].sum()
print(f"\n--- Total spoilage cost: {total_spoilage_cost:,.0f} ---")
print(f"\n--- Spoilage by category ---")
print(spoilage_by_category.round(0))


--- Total spoilage cost: 6,199,272 ---

--- Spoilage by category ---
category
Perishables            4,678,592.00
Beverages                669,932.00
Personal Care            610,592.00
Packaged Foods           240,155.00
Household Essentials           0.00
Name: spoilage_cost, dtype: float64


In [21]:
#  SIMPLE DEMAND FORECAST (moving average) VS ACTUAL 
# pick the highest-volume product for a clean illustrative example
top_product_id = df.groupby("product_id")["units_sold"].sum().idxmax()
top_product_name = products.loc[products["product_id"] == top_product_id, "product_name"].iloc[0]

#  weekly sold units --> prod_ts
prod_ts = (df[df["product_id"] == top_product_id]
           .groupby("week_start")["units_sold"].sum()
           .sort_index())

window = 4
forecast = prod_ts.rolling(window=window).mean().shift(1)  # forecast using prior 4 weeks, no lookahead
valid = pd.DataFrame({"actual": prod_ts, "forecast": forecast}).dropna()

mape = (np.abs(valid["actual"] - valid["forecast"]) / valid["actual"].replace(0, np.nan)).mean() * 100
print(f"\n--- Demand forecast (4-week moving average) for '{top_product_name}' ---")
print(f"MAPE: {mape:.1f}%")


--- Demand forecast (4-week moving average) for 'Energy Drink Can' ---
MAPE: 7.2%


In [23]:
#  CHARTS :
plt.style.use("seaborn-v0_8-whitegrid")

# a. Stockout rate by category
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(stockout_by_category.index, stockout_by_category.values, color="#C73E1D")
ax.set_title("Stockout Rate by Category (%)", fontsize=14, fontweight="bold")
ax.set_xlabel("Stockout Rate (%)")
plt.tight_layout()
plt.savefig("images/stockout_rate_by_category.png", dpi=150)
plt.close()

# b. Stockout rate by region
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(stockout_by_region.index, stockout_by_region.values, color="#2E86AB")
ax.set_title("Stockout Rate by Region (%)", fontsize=14, fontweight="bold")
ax.set_ylabel("Stockout Rate (%)")
plt.tight_layout()
plt.savefig("images/stockout_rate_by_region.png", dpi=150)
plt.close()

# c. Inventory turnover by category
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(turnover_by_category.index, turnover_by_category.values, color="#F18F01")
ax.set_title("Inventory Turnover by Category (annualized)", fontsize=14, fontweight="bold")
ax.set_xlabel("Turnover Ratio (COGS / Avg Inventory Value)")
plt.tight_layout()
plt.savefig("images/inventory_turnover_by_category.png", dpi=150)
plt.close()

# d. Supplier lead time vs stockout rate scatter
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(supplier_perf["avg_lead_time_days"], supplier_perf["stockout_rate_pct"],
           s=100, color="#A23B72")
for sid, row in supplier_perf.iterrows():
    ax.annotate(sid, (row["avg_lead_time_days"], row["stockout_rate_pct"]),
                fontsize=8, xytext=(4, 4), textcoords="offset points")
ax.set_title("Supplier Lead Time vs. Stockout Rate", fontsize=14, fontweight="bold")
ax.set_xlabel("Avg Lead Time (days)")
ax.set_ylabel("Stockout Rate (%)")
plt.tight_layout()
plt.savefig("images/supplier_lead_time_vs_stockouts.png", dpi=150)
plt.close()

# e. Demand forecast vs actual
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(valid.index, valid["actual"], label="Actual", color="#2E86AB", marker="o", markersize=3)
ax.plot(valid.index, valid["forecast"], label="Forecast (4-wk moving avg)", color="#C73E1D", linestyle="--")
ax.set_title(f"Demand Forecast vs. Actual — {top_product_name}", fontsize=14, fontweight="bold")
ax.set_ylabel("Units Sold (weekly)")
plt.xticks(rotation=45, ha="right")
ax.legend()
plt.tight_layout()
plt.savefig("images/demand_forecast_vs_actual.png", dpi=150)
plt.close()

# f. Spoilage cost by category
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(spoilage_by_category.index, spoilage_by_category.values, color="#5B8C5A")
ax.set_title("Total Spoilage Cost by Category", fontsize=14, fontweight="bold")
ax.set_ylabel("Spoilage Cost")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("images/spoilage_cost_by_category.png", dpi=150)
plt.close()

print("\n All charts saved to images/")


 All charts saved to images/


In [25]:
# SAVE SUMMARY :
summary = {
    "overall_fill_rate_pct": round(overall_fill_rate, 1),
    "overall_stockout_rate_pct": round(overall_stockout_rate, 1),
    "total_lost_sales_value": round(total_lost_sales_value, 0),
    "total_spoilage_cost": round(total_spoilage_cost, 0),
    "worst_stockout_category": stockout_by_category.idxmax(),
    "worst_stockout_category_rate_pct": round(stockout_by_category.max(), 1),
    "worst_stockout_region": stockout_by_region.idxmax(),
    "lead_time_stockout_corr": round(lead_time_stockout_corr, 2),
    "reliability_stockout_corr": round(reliability_stockout_corr, 2),
    "demand_forecast_example_product": top_product_name,
    "demand_forecast_mape_pct": round(mape, 1),
}
pd.Series(summary).to_csv("supply_data/summary_metrics.csv")
print("\nSummary:")
for key, value in summary.items():
    print(f"{key}: {value}\n")


Summary:
overall_fill_rate_pct: 98.2

overall_stockout_rate_pct: 1.8

total_lost_sales_value: 2123038.0

total_spoilage_cost: 6199272.0

worst_stockout_category: Packaged Foods

worst_stockout_category_rate_pct: 2.2

worst_stockout_region: West

lead_time_stockout_corr: -0.45

reliability_stockout_corr: -0.44

demand_forecast_example_product: Energy Drink Can

demand_forecast_mape_pct: 7.2



In [26]:
#  POWER-BI-READY EXPORT
# a single flat, joined table is the easiest starting point for a Power BI model
powerbi_export = df[[
    "store_id", "store_name", "region", "store_type",
    "product_id", "product_name", "category",
    "week_start", "opening_stock", "units_received", "spoilage_units",
    "units_sold", "closing_stock", "stockout_flag", "lost_sales_units",
    "lost_sales_value", "reorder_point", "safety_stock",
    "supplier_id", "avg_lead_time_days", "reliability_score",
    "unit_cost", "unit_price", "inventory_value", "cogs", "spoilage_cost",
]]
powerbi_export.to_csv("supply_data/powerbi_dataset.csv", index=False)
print("Saved Power BI-ready flat dataset to supply_data/powerbi_dataset.csv")

Saved Power BI-ready flat dataset to supply_data/powerbi_dataset.csv
